# Pipeline — step-by-step audit

Runs each `src/` stage in sequence, inspects data state, and runs asserts we define together.

All transformations live in `src/`; here we only import, run, and verify.

**Prerequisite:** `data/raw/` with the three JSON files.

## 0. Setup

Config, Spark session, and inspection helper.

In [2]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

from src.attribution import add_recurrence_flag, attribute, build_label
from src.clean import normalize_profile
from src.config import load
from src.cost import add_reward_cost
from src import contract
from src.features import build
from src.io import parse_events, read_offers, read_profile
from src.pipeline import _campaign_wave, _treatment, build_spark

cfg = load()
spark = build_spark(cfg, app_name="ifood-uplift-audit")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/12 01:12:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
def snapshot(df: DataFrame, name: str, sample: int = 3) -> None:
    """Row count, schema, and sample rows — inspection only."""
    n = df.count()
    print(f"\n=== {name} ===")
    print(f"rows: {n:,}  |  columns: {len(df.columns)}")
    df.printSchema()
    if n:
        df.show(sample, truncate=60)

## 1. Base data

Reads the three JSON files via `src/io` and summarizes volume, coverage, and time horizon.

In [4]:
events = parse_events(spark, cfg).cache()
offers = read_offers(spark, cfg).cache()
profile_raw = read_profile(spark, cfg).cache()

n_events = events.count()
n_offers = offers.count()
n_profile = profile_raw.count()
n_customers_events = events.select("account_id").distinct().count()
n_customers_profile = profile_raw.select("id").distinct().count()
t_min, t_max = events.agg(F.min("time"), F.max("time")).first()

print("=== Base data — overall status ===")
print(f"transactions → {cfg.transactions_path}")
print(f"profile      → {cfg.profile_path}")
print(f"offers       → {cfg.offers_path}")
print()
print(f"{'source':<22} {'rows':>10}  {'customers':>10}")
print(f"{'parsed events':<22} {n_events:>10,}  {n_customers_events:>10,}")
print(f"{'profile (id)':<22} {n_profile:>10,}  {n_customers_profile:>10,}")
print(f"{'catalog':<22} {n_offers:>10,}  {'—':>10}")
print(f"\ntime horizon: t = {t_min:.2f} … {t_max:.2f}")

snapshot(events, "parsed events")
print("count by event type:")
events.groupBy("event").count().orderBy("event").show()

snapshot(offers, "offer catalog")
print("offers by type:")
offers.groupBy("offer_type").count().orderBy("offer_type").show()

snapshot(profile_raw, "customer profile")

=== Base data — overall status ===
transactions → data/raw/transactions.json
profile      → data/raw/profile.json
offers       → data/raw/offers.json

source                       rows   customers
parsed events             306,534      17,000
profile (id)               17,000      17,000
catalog                        10           —

time horizon: t = 0.00 … 29.75

=== parsed events ===
rows: 306,534  |  columns: 6
root
 |-- account_id: string (nullable = true)
 |-- event: string (nullable = true)
 |-- time: double (nullable = true)
 |-- offer_ref: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- reward: double (nullable = true)

+--------------------------------+--------------+----+--------------------------------+------+------+
|                      account_id|         event|time|                       offer_ref|amount|reward|
+--------------------------------+--------------+----+--------------------------------+------+------+
|78afa995795e4d85b5d9ceeca43f5fef|off

### 1.1 Load checks

- Check that the three source files exist on disk
- Confirm that all sources contain at least one row
- Verify that the schema (column names) of each DataFrame matches the expectation

In [5]:
EVENT_TYPES = {"offer completed", "offer received", "offer viewed", "transaction"}
OFFER_LIFECYCLE = ("offer received", "offer viewed", "offer completed")

assert cfg.transactions_path.is_file()
assert cfg.profile_path.is_file()
assert cfg.offers_path.is_file()

assert n_events > 0 and n_offers > 0 and n_profile > 0

assert set(events.columns) == {"account_id", "event", "time", "offer_ref", "amount", "reward"}
assert set(offers.columns) == {"channels", "discount_value", "duration", "id", "min_value", "offer_type"}
assert set(profile_raw.columns) == {"age", "credit_card_limit", "gender", "id", "registered_on"}


print("base data load checks passed")

base data load checks passed


## 2. Profile normalization
- Apply the `clean.normalize_profile` function to the raw customer profile DataFrame (`profile_raw`)
- Standardize column names
- Handle missing values
- Flag records missing key identity information
- Convert the original id to `account_id`
- Normalize gender values
- Compute `tenure_days` (customer relationship length)
- The resulting cleaned and enriched DataFrame is used for all subsequent analyses

In [6]:
profile = normalize_profile(profile_raw, cfg).cache()
n_profile_norm = profile.count()

tenure_min, tenure_max = profile.agg(F.min("tenure_days"), F.max("tenure_days")).first()
n_identity_missing = profile.filter(F.col("identity_missing") == 1).count()
n_unknown_gender = profile.filter(F.col("gender") == "unknown").count()

print("=== Profile normalization ===")
print(f"rows in → rows out: {n_profile:,} → {n_profile_norm:,}")
print(f"identity_missing: {n_identity_missing:,}")
print(f"gender = unknown:   {n_unknown_gender:,}")
print(f"tenure_days: {tenure_min:,} … {tenure_max:,}")

snapshot(profile, "normalized profile")

=== Profile normalization ===
rows in → rows out: 17,000 → 17,000
identity_missing: 2,175
gender = unknown:   2,175
tenure_days: 0 … 1,823

=== normalized profile ===
rows: 17,000  |  columns: 6
root
 |-- account_id: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = false)
 |-- credit_card_limit: double (nullable = true)
 |-- identity_missing: integer (nullable = false)
 |-- tenure_days: integer (nullable = true)

+--------------------------------+----+-------+-----------------+----------------+-----------+
|                      account_id| age| gender|credit_card_limit|identity_missing|tenure_days|
+--------------------------------+----+-------+-----------------+----------------+-----------+
|68be06ca386d4c31939f3a4f0e3dd783|NULL|unknown|             NULL|               1|        529|
|0610b486422d4921ae7d2bf64640c50b|  55|      F|         112000.0|               0|        376|
|38fe809add3b4fcf9315a9694bb96ff5|NULL|unknown|             NULL|  

### 2.1 Normalization checks

- Asserts that the normalized profile row count matches the raw profile row count
- Checks that the profile columns match the expected schema
- Ensures there are no null account_id values
- Validates that all account_id values are unique
- Asserts that identity_missing column contains only 0 or 1
- Confirms that if identity_missing is 1, age is always null
- Checks that if identity_missing is 0, age is never equal to the sentinel value
- Ensures there are no null values in the gender or tenure_days columns
- Verifies that the number of rows with age equal to the sentinel value in the raw profile matches the number of rows flagged as identity_missing in the normalized profile

In [7]:
n_sentinel_raw = profile_raw.filter(F.col("age") == cfg.age_sentinel).count()

assert n_profile_norm == n_profile
assert set(profile.columns) == {
    "account_id", "age", "gender", "credit_card_limit", "identity_missing", "tenure_days",
}

assert profile.filter(F.col("account_id").isNull()).count() == 0
assert profile.select("account_id").distinct().count() == n_profile_norm
assert profile.filter(~F.col("identity_missing").isin(0, 1)).count() == 0
assert profile.filter((F.col("identity_missing") == 1) & F.col("age").isNotNull()).count() == 0
assert profile.filter((F.col("identity_missing") == 0) & (F.col("age") == cfg.age_sentinel)).count() == 0
assert profile.filter(F.col("gender").isNull()).count() == 0
assert profile.filter(F.col("tenure_days").isNull()).count() == 0
assert n_sentinel_raw == n_identity_missing

# A fonte bruta não é mais usada; o perfil normalizado segue até o join final.
profile_raw.unpersist()

print("profile normalization checks passed")

profile normalization checks passed


## 3. Attribution

 Step-by-step overview of what the `attribution.attribute` function does:
 
 - For each "offer received" event in the event log, creates one output row.
 - Looks for the corresponding "offer viewed" event for the same account and offer.
   - If found, assigns the view timestamp (`view_time`) to the row; otherwise, leaves it null.
 - Searches for any transactions ("txn") that are eligible to be associated with the offer (e.g., meet time, account, and offer criteria).
   - If there is exactly one matching transaction, assigns its details (`assigned_txn_count`, `assigned_txn_amount_sum`, etc.) to the row.
   - If there is no matching transaction, sets transaction-related fields accordingly (usually zeros or nulls).
 - Ensures each row contains:
   - account ID
   - offer ID
   - offer received time
   - offer valid-until time window
   - view timestamp, if any
   - transaction assignment count (should be 0 or 1)
   - sum of assigned transaction amounts
   - timestamp of first assigned transaction
 - Produces a table where each row represents a unique "offer received" event and its downstream attribution status (view, transaction).

In [8]:
attributed = attribute(events, offers, cfg).cache()
n_attributed = attributed.count()
n_received = events.filter(F.col("event") == "offer received").count()
n_with_view = attributed.filter(F.col("view_time").isNotNull()).count()
n_with_txn = attributed.filter(F.col("assigned_txn_count") == 1).count()

print("=== Attribution ===")
print(f"offer received → attributed rows: {n_received:,} → {n_attributed:,}")
print(f"with view:        {n_with_view:,}")
print(f"with transaction: {n_with_txn:,}")

snapshot(attributed, "attributed grain")

Premissa 1 violada em 15721 transação(ões): mais de uma oferta ativa no intervalo; prioridade 'earliest_received' aplicada.


=== Attribution ===
offer received → attributed rows: 76,277 → 76,277
with view:        54,654
with transaction: 34,004

=== attributed grain ===
rows: 76,277  |  columns: 8
root
 |-- account_id: string (nullable = true)
 |-- offer_id: string (nullable = true)
 |-- received_time: double (nullable = true)
 |-- valid_until: double (nullable = true)
 |-- view_time: double (nullable = true)
 |-- assigned_txn_count: long (nullable = false)
 |-- assigned_txn_amount_sum: double (nullable = false)
 |-- first_assigned_txn_time: double (nullable = true)

+--------------------------------+--------------------------------+-------------+-----------+---------+------------------+-----------------------+-----------------------+
|                      account_id|                        offer_id|received_time|valid_until|view_time|assigned_txn_count|assigned_txn_amount_sum|first_assigned_txn_time|
+--------------------------------+--------------------------------+-------------+-----------+---------+----

### 3.1 Attribution checks

- Checks that the number of attributed rows equals the number of "offer received" events
- Ensures the set of columns matches the expected schema for the attributed data
- Verifies that each (account_id, offer_id, received_time) combination is unique
- Confirms there are no null values in account_id or offer_id fields
- Checks that all offer_ids are present in the offers catalog
- Validates that assigned_txn_count is always 0 or 1
- Ensures view_time, if present, is between received_time and valid_until
- Checks that when assigned_txn_count is 0, the transaction sum is 0 and no transaction time is set
- Makes sure that if assigned_txn_count is 1, then the assigned_txn_amount_sum is positive, the transaction time is present, and the transaction time falls within the offer window

In [9]:
catalog_ids = {r["id"] for r in offers.select("id").collect()}

assert n_attributed == n_received
assert set(attributed.columns) == {
    "account_id",
    "offer_id",
    "received_time",
    "valid_until",
    "view_time",
    "assigned_txn_count",
    "assigned_txn_amount_sum",
    "first_assigned_txn_time",
}

assert attributed.select("account_id", "offer_id", "received_time").distinct().count() == n_attributed
assert attributed.filter(F.col("account_id").isNull() | F.col("offer_id").isNull()).count() == 0
assert attributed.filter(~F.col("offer_id").isin(*catalog_ids)).count() == 0
assert attributed.filter(~F.col("assigned_txn_count").isin(0, 1)).count() == 0

assert attributed.filter(
    F.col("view_time").isNotNull()
    & (
        (F.col("view_time") < F.col("received_time"))
        | (F.col("view_time") > F.col("valid_until"))
    )
).count() == 0

assert attributed.filter(
    (F.col("assigned_txn_count") == 0)
    & (
        (F.col("assigned_txn_amount_sum") != 0)
        | F.col("first_assigned_txn_time").isNotNull()
    )
).count() == 0

assert attributed.filter(
    (F.col("assigned_txn_count") == 1)
    & (
        (F.col("assigned_txn_amount_sum") <= 0)
        | F.col("first_assigned_txn_time").isNull()
        | (F.col("first_assigned_txn_time") < F.col("received_time"))
        | (F.col("first_assigned_txn_time") > F.col("valid_until"))
    )
).count() == 0

print("attribution checks passed")

attribution checks passed


## 4. Label

- Call the function `build_label` from the `attribution` module to generate two new columns: `converted` and `conversion_value`
- Use as input the previously attributed DataFrame and the configuration object (`cfg`)
- The output DataFrame now includes a label (`converted`) indicating whether a conversion occurred for each row
- The resulting `conversion_value` column describes the value associated with the conversion (if any)
- This step allows subsequent analysis to distinguish between converted and non-converted offers, and to quantify successful conversions

In [10]:
labeled = build_label(attributed, cfg).cache()
n_labeled = labeled.count()
n_converted = labeled.filter(F.col("converted") == 1).count()
n_converted_no_view = labeled.filter(F.col("view_time").isNull() & (F.col("converted") == 1)).count()

print("=== Label ===")
print(f"attributed → labeled rows: {n_attributed:,} → {n_labeled:,}")
print(f"converted:              {n_converted:,}")
print(f"converted without view: {n_converted_no_view:,}")

snapshot(labeled, "labeled grain")

=== Label ===
attributed → labeled rows: 76,277 → 76,277
converted:              34,004
converted without view: 7,140

=== labeled grain ===
rows: 76,277  |  columns: 10
root
 |-- account_id: string (nullable = true)
 |-- offer_id: string (nullable = true)
 |-- received_time: double (nullable = true)
 |-- valid_until: double (nullable = true)
 |-- view_time: double (nullable = true)
 |-- assigned_txn_count: long (nullable = false)
 |-- assigned_txn_amount_sum: double (nullable = false)
 |-- first_assigned_txn_time: double (nullable = true)
 |-- converted: integer (nullable = false)
 |-- conversion_value: double (nullable = false)

+--------------------------------+--------------------------------+-------------+-----------+---------+------------------+-----------------------+-----------------------+---------+----------------+
|                      account_id|                        offer_id|received_time|valid_until|view_time|assigned_txn_count|assigned_txn_amount_sum|first_assigned_tx

### 4.1 Label checks

- Validates the structure of the `converted` and `conversion_value` columns.
- Checks that the set of columns after labeling is as expected.
- Ensures each combination of `account_id`, `offer_id`, and `received_time` is unique.
- Guarantees that the `converted` column only contains the values 0 or 1.
- Asserts that only rows with exactly 1 assigned transaction are marked as converted.
- Asserts that only rows with zero assigned transactions are marked as not converted.
- Confirms that conversion value is zero when not converted.
- Confirms that conversion value is positive when converted.
- Asserts that the conversion value matches the assigned transaction amount sum for converted rows.
- Ensures that the conversion value for converted rows is always equal to or greater than the minimum allowed offer value.

In [11]:
offers_min = offers.select(F.col("id").alias("offer_id"), F.col("min_value").cast("double"))

assert n_labeled == n_attributed
assert set(labeled.columns) == set(attributed.columns) | {"converted", "conversion_value"}

assert labeled.select("account_id", "offer_id", "received_time").distinct().count() == n_labeled
assert labeled.filter(~F.col("converted").isin(0, 1)).count() == 0

assert labeled.filter((F.col("converted") == 1) & (F.col("assigned_txn_count") != 1)).count() == 0
assert labeled.filter((F.col("converted") == 0) & (F.col("assigned_txn_count") != 0)).count() == 0

assert labeled.filter((F.col("converted") == 0) & (F.col("conversion_value") != 0)).count() == 0
assert labeled.filter((F.col("converted") == 1) & (F.col("conversion_value") <= 0)).count() == 0
assert labeled.filter(
    (F.col("converted") == 1) & (F.col("conversion_value") != F.col("assigned_txn_amount_sum"))
).count() == 0

assert (
    labeled.join(offers_min, "offer_id")
    .filter((F.col("converted") == 1) & (F.col("conversion_value") < F.col("min_value")))
    .count()
    == 0
)

# A atribuição já foi materializada no label e não volta a ser consultada.
attributed.unpersist()

print("label checks passed")

label checks passed


## 5. Recurrence flag and features

| Feature                      | Type         | Brief Explanation                                                                                          |
|------------------------------|--------------|-----------------------------------------------------------------------------------------------------------|
| discount_value               | Numeric      | The discount amount for the offer.                                                                        |
| min_value                    | Numeric      | Minimum spend required to qualify for the offer.                                                          |
| duration                     | Integer      | Duration of the offer (in days).                                                                          |
| n_channels                   | Integer      | Number of channels through which the offer was sent.                                                      |
| channel_web                  | Binary (0/1) | Indicates if the offer was available via the Web channel.                                                 |
| channel_email                | Binary (0/1) | Indicates if the offer was sent via Email.                                                                |
| channel_mobile               | Binary (0/1) | Indicates if the offer was sent via Mobile app.                                                           |
| channel_social               | Binary (0/1) | Indicates if the offer was sent via Social Media.                                                         |
| discount_to_minvalue_ratio   | Numeric      | The ratio of discount value to minimum spend.                                                             |
| n_concurrent_offers          | Integer      | Number of other offers active for the customer at the reception time of this offer.                       |
| hist_spend_total             | Numeric      | Total spend by the customer before receiving the offer.                                                   |
| hist_txn_count               | Integer      | Count of transactions before receiving the offer.                                                         |
| hist_avg_ticket              | Numeric      | Average transaction ticket amount before receiving the offer.                                              |
| hist_spend_std               | Numeric      | Standard deviation of spend before receiving the offer.                                                   |
| hist_recency_days            | Numeric      | Number of days since last transaction, relative to offer receipt.                                         |
| hist_frequency               | Numeric      | Frequency of transactions before offer receipt.                                                           |
| hist_spend_trend             | Numeric      | Trend in customer spend over time before receiving the offer.                                             |
| hist_offers_received         | Integer      | Total count of previous offers received before this offer.                                                |
| hist_offers_viewed           | Integer      | Total count of previous offers viewed before this offer.                                                  |
| hist_offers_completed        | Integer      | Total count of previous offers completed before this offer.                                               |
| hist_offers_received_bogo    | Integer      | Count of previous 'buy one get one' (BOGO) offers received.                                               |
| hist_offers_received_discount| Integer      | Count of previous discount-type offers received.                                                          |
| hist_offers_received_info    | Integer      | Count of previous informational offers received.                                                          |
| hist_view_rate               | Numeric      | Ratio of offers viewed to offers received before this offer.                                              |
| hist_conv_rate_bogo          | Numeric      | Conversion rate for BOGO offers among previous offers.                                                    |
| hist_conv_rate_discount      | Numeric      | Conversion rate for discount offers among previous offers.                                                |
| hist_completed_unseen_flag   | Binary (0/1) | Whether the customer completed an offer without viewing it previously (in the past).                      |
| hist_time_view_to_conv       | Numeric      | Average time from offer view to conversion for past offers (in days).                                     |


- Apply `add_recurrence_flag` to the labeled grain **first**, exactly as `assemble_processed` (before features): `is_recurrent` is label-derived and rides through the feature and cost stages.
- Call `features.build` with the parsed event log, the recurrence-flagged grain, and offer catalog.
- Aggregate each customer's transaction history using only events with `time < received_time`.
- Count prior offer events (received, viewed, completed) and derive view and conversion rates by type.
- Join catalog fields for the current offer (discount, minimum spend, duration, channels).
- Count how many other offers were active for the same customer at receipt time (`n_concurrent_offers`).
- Flag whether the customer ever completed an offer without a prior view (`hist_completed_unseen_flag`).
- Compute average lag from view to completion on past offers (`hist_time_view_to_conv`).
- Left-join every feature block so rows without history survive with zeroed counters.
- Preserve the full labeled grain; append `is_recurrent` and 28 feature columns.

In [12]:
# assemble_processed aplica add_recurrence_flag ao grão rotulado ANTES das
# features: is_recurrent é derivada do label e flui pelas features e pelo custo.
with_recurrence = add_recurrence_flag(labeled, cfg)
n_recurrent = with_recurrence.filter(F.col("is_recurrent") == 1).count()

featured = build(events, with_recurrence, offers, cfg).cache()
n_featured = featured.count()
n_feature_cols = len(featured.columns) - len(with_recurrence.columns)
n_zero_hist_txn = featured.filter(F.col("hist_txn_count") == 0).count()

print("=== Recurrence flag and features ===")
print(f"is_recurrent=1:                      {n_recurrent:,}")
print(f"labeled+recurrence → featured rows:  {n_labeled:,} → {n_featured:,}")
print(f"feature columns added:               {n_feature_cols}")
print(f"rows with zero txn history:          {n_zero_hist_txn:,}")

snapshot(featured, "featured grain")

26/07/12 01:12:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


=== Recurrence flag and features ===
is_recurrent=1:                      9,879
labeled+recurrence → featured rows:  76,277 → 76,277
feature columns added:               28
rows with zero txn history:          20,952

=== featured grain ===
rows: 76,277  |  columns: 39
root
 |-- account_id: string (nullable = true)
 |-- offer_id: string (nullable = true)
 |-- received_time: double (nullable = true)
 |-- valid_until: double (nullable = true)
 |-- view_time: double (nullable = true)
 |-- assigned_txn_count: long (nullable = false)
 |-- assigned_txn_amount_sum: double (nullable = false)
 |-- first_assigned_txn_time: double (nullable = true)
 |-- converted: integer (nullable = false)
 |-- conversion_value: double (nullable = false)
 |-- is_recurrent: integer (nullable = false)
 |-- discount_value: long (nullable = true)
 |-- min_value: long (nullable = true)
 |-- duration: double (nullable = true)
 |-- n_channels: integer (nullable = true)
 |-- channel_web: integer (nullable = true)
 |-- c

### 5.1 Feature checks

- Asserts that the featured row count matches the labeled row count.
- Verifies that every labeled column is still present and 28 feature columns were added.
- Confirms `is_recurrent` was appended, is binary (0/1), and is 1 only on converted rows.
- Ensures the `(account_id, offer_id, received_time)` grain remains unique.
- Checks that integer history and channel flags are non-null and non-negative.
- Confirms `hist_completed_unseen_flag` and channel indicators are binary (0 or 1).
- Validates that non-nullable historical spend and offer-count columns have no nulls.
- Ensures historical rates are never negative.
- Confirms `hist_recency_days`, when present, is strictly positive (last transaction strictly before receipt).

In [13]:
ADDED_COLS = {
    "discount_value", "min_value", "duration", "n_channels",
    "channel_web", "channel_email", "channel_mobile", "channel_social",
    "discount_to_minvalue_ratio", "n_concurrent_offers",
    "hist_spend_total", "hist_txn_count", "hist_avg_ticket", "hist_spend_std",
    "hist_recency_days", "hist_frequency", "hist_spend_trend",
    "hist_offers_received", "hist_offers_viewed", "hist_offers_completed",
    "hist_offers_received_bogo", "hist_offers_received_discount", "hist_offers_received_info",
    "hist_view_rate", "hist_conv_rate_bogo", "hist_conv_rate_discount",
    "hist_completed_unseen_flag", "hist_time_view_to_conv",
}
HIST_INT_COLS = [
    "hist_txn_count", "hist_offers_received", "hist_offers_viewed", "hist_offers_completed",
    "hist_offers_received_bogo", "hist_offers_received_discount", "hist_offers_received_info",
    "hist_completed_unseen_flag", "n_concurrent_offers", "n_channels",
    "channel_web", "channel_email", "channel_mobile", "channel_social",
]
HIST_DBL_NONNULL = [
    "hist_spend_total", "hist_avg_ticket", "hist_spend_std", "hist_frequency", "hist_spend_trend",
    "hist_view_rate", "hist_conv_rate_bogo", "hist_conv_rate_discount",
    "discount_to_minvalue_ratio",
]

assert n_featured == n_labeled
assert set(labeled.columns) <= set(featured.columns)
assert ADDED_COLS <= set(featured.columns)
assert featured.select("account_id", "offer_id", "received_time").distinct().count() == n_featured

assert "is_recurrent" in set(featured.columns)
assert featured.filter(~F.col("is_recurrent").isin(0, 1)).count() == 0
assert featured.filter((F.col("converted") == 0) & (F.col("is_recurrent") != 0)).count() == 0
assert featured.filter((F.col("is_recurrent") == 1) & (F.col("converted") != 1)).count() == 0

for col in HIST_INT_COLS:
    assert featured.filter(F.col(col).isNull()).count() == 0
    assert featured.filter(F.col(col) < 0).count() == 0

assert featured.filter(~F.col("hist_completed_unseen_flag").isin(0, 1)).count() == 0
for col in ("channel_web", "channel_email", "channel_mobile", "channel_social"):
    assert featured.filter(~F.col(col).isin(0, 1)).count() == 0

for col in HIST_DBL_NONNULL:
    assert featured.filter(F.col(col).isNull()).count() == 0

assert featured.filter(
    (F.col("hist_view_rate") < 0)
    | (F.col("hist_conv_rate_bogo") < 0)
    | (F.col("hist_conv_rate_discount") < 0)
).count() == 0

assert featured.filter(F.col("hist_recency_days").isNotNull() & (F.col("hist_recency_days") <= 0)).count() == 0

# O log de eventos e o grão rotulado não são usados depois das features.
events.unpersist()
labeled.unpersist()

print("feature checks passed")

feature checks passed


## 6. Reward cost

- Call `cost.add_reward_cost` on the featured grain and the offer catalog.
- Join `offer_type` from the catalog onto each row.
- Set `reward_cost` to the catalog `discount_value` when the row converted on a rewardable offer (`bogo` or `discount`).
- Set `reward_cost` to zero when the row did not convert or the offer is `informational`.
- Charge the discount on any real conversion, whether or not the customer viewed the offer.
- Preserve row count and grain; append `offer_type` and `reward_cost`.

In [14]:
priced = add_reward_cost(featured, offers, cfg).cache()
n_priced = priced.count()
n_with_cost = priced.filter(F.col("reward_cost") > 0).count()
total_reward_cost = priced.agg(F.sum("reward_cost")).first()[0]

print("=== Reward cost ===")
print(f"featured → priced rows: {n_featured:,} → {n_priced:,}")
print(f"rows with reward_cost > 0: {n_with_cost:,}")
print(f"total reward_cost:         R$ {total_reward_cost:,.2f}")

snapshot(priced, "priced grain")

=== Reward cost ===
featured → priced rows: 76,277 → 76,277
rows with reward_cost > 0: 25,996
total reward_cost:         R$ 131,568.00

=== priced grain ===
rows: 76,277  |  columns: 41
root
 |-- offer_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- received_time: double (nullable = true)
 |-- valid_until: double (nullable = true)
 |-- view_time: double (nullable = true)
 |-- assigned_txn_count: long (nullable = false)
 |-- assigned_txn_amount_sum: double (nullable = false)
 |-- first_assigned_txn_time: double (nullable = true)
 |-- converted: integer (nullable = false)
 |-- conversion_value: double (nullable = false)
 |-- is_recurrent: integer (nullable = false)
 |-- discount_value: long (nullable = true)
 |-- min_value: long (nullable = true)
 |-- duration: double (nullable = true)
 |-- n_channels: integer (nullable = true)
 |-- channel_web: integer (nullable = true)
 |-- channel_email: integer (nullable = true)
 |-- channel_mobile: integer (nullable = tru

### 6.1 Reward cost checks

- Asserts that the priced row count matches the featured row count.
- Verifies that `offer_type` and `reward_cost` were added and the grain stayed unique.
- Ensures `reward_cost` is never negative.
- Confirms `reward_cost` is zero whenever the row did not convert.
- Confirms `reward_cost` is zero for `informational` offers, even when converted.
- Checks that any row with positive `reward_cost` converted on a `bogo` or `discount` offer.
- Validates that rewardable conversions carry `reward_cost` equal to the catalog `discount_value`.

In [15]:
assert n_priced == n_featured
assert {"offer_type", "reward_cost"} <= set(priced.columns)
assert priced.select("account_id", "offer_id", "received_time").distinct().count() == n_priced

assert priced.filter(F.col("reward_cost") < 0).count() == 0
assert priced.filter((F.col("converted") == 0) & (F.col("reward_cost") != 0)).count() == 0
assert priced.filter((F.col("offer_type") == "informational") & (F.col("reward_cost") != 0)).count() == 0

assert priced.filter(
    (F.col("reward_cost") > 0)
    & ((F.col("converted") != 1) | F.col("offer_type").isin("informational"))
).count() == 0

assert priced.filter(
    (F.col("converted") == 1)
    & F.col("offer_type").isin("bogo", "discount")
    & (F.col("reward_cost") != F.col("discount_value").cast("double"))
).count() == 0

# O catálogo e as features já foram incorporados ao grão com custo.
offers.unpersist()
featured.unpersist()

print("reward cost checks passed")

reward cost checks passed


## 7. Profile join and derived columns

- Left-join the normalized profile on `account_id` (`age`, `gender`, `credit_card_limit`, `identity_missing`, `tenure_days`).
- Derive `treatment` with `pipeline._treatment` (`treatment=1` iff `view_time` is not null — exposure, not receipt).
- Derive `campaign_wave` with `pipeline._campaign_wave` (0-based dense rank of distinct `received_time`).
- Preserve row count and grain through the join and both derivations.

In [16]:
# Espelha o fim de assemble_processed: join do perfil, depois treatment e
# campaign_wave pelas mesmas funções de src/pipeline (nada reimplementado aqui).
enriched = priced.join(profile, on="account_id", how="left")
with_derived = _campaign_wave(_treatment(enriched)).cache()
n_derived = with_derived.count()

# O resultado final já está materializado; suas duas entradas podem sair do cache.
priced.unpersist()
profile.unpersist()

n_treated = with_derived.filter(F.col("treatment") == 1).count()
n_control = with_derived.filter(F.col("treatment") == 0).count()
n_waves = with_derived.select("campaign_wave").distinct().count()
n_converted_control = with_derived.filter((F.col("treatment") == 0) & (F.col("converted") == 1)).count()

print("=== Profile join and derived columns ===")
print(f"priced → enriched rows: {n_priced:,} → {n_derived:,}")
print(f"treatment=1 (saw):      {n_treated:,}")
print(f"treatment=0 (no view):  {n_control:,}")
print(f"converted in control:   {n_converted_control:,}")
print(f"campaign waves:         {n_waves}")

print("\ncampaign_wave by received_time:")
(
    with_derived.select("received_time", "campaign_wave")
    .distinct()
    .orderBy("received_time")
    .show()
)

snapshot(with_derived, "enriched grain")

26/07/12 01:13:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/12 01:13:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/12 01:13:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/12 01:13:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/12 01:13:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/12 01:13:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/12 0

=== Profile join and derived columns ===
priced → enriched rows: 76,277 → 76,277
treatment=1 (saw):      54,654
treatment=0 (no view):  21,623
converted in control:   7,140
campaign waves:         6

campaign_wave by received_time:
+-------------+-------------+
|received_time|campaign_wave|
+-------------+-------------+
|          0.0|            0|
|          7.0|            1|
|         14.0|            2|
|         17.0|            3|
|         21.0|            4|
|         24.0|            5|
+-------------+-------------+


=== enriched grain ===
rows: 76,277  |  columns: 48
root
 |-- received_time: double (nullable = true)
 |-- account_id: string (nullable = true)
 |-- offer_id: string (nullable = true)
 |-- valid_until: double (nullable = true)
 |-- view_time: double (nullable = true)
 |-- assigned_txn_count: long (nullable = false)
 |-- assigned_txn_amount_sum: double (nullable = false)
 |-- first_assigned_txn_time: double (nullable = true)
 |-- converted: integer (nullable = fa

### 7.1 Assembly checks

- Asserts that row count is preserved from priced through the profile join and both derivations.
- Confirms `is_recurrent`, `treatment`, `campaign_wave`, and the profile columns are present, and the grain stayed unique.
- Ensures every row has a matching profile (`age` or `identity_missing` present).
- Validates `treatment` is 1 iff `view_time` is not null.
- Validates `campaign_wave` is the 0-based dense rank of distinct `received_time` (contiguous from zero, one wave per dispatch).
- Confirms the number of distinct waves equals the number of distinct `received_time` values.

In [17]:
PROFILE_COLS = {"age", "gender", "credit_card_limit", "identity_missing", "tenure_days"}

assert n_derived == n_priced
assert {"is_recurrent", "treatment", "campaign_wave"} <= set(with_derived.columns)
assert PROFILE_COLS <= set(with_derived.columns)
assert with_derived.select("account_id", "offer_id", "received_time").distinct().count() == n_derived

assert with_derived.filter(F.col("age").isNull() & (F.col("identity_missing") == 0)).count() == 0

assert with_derived.filter(
    (F.col("treatment") == 1) != F.col("view_time").isNotNull()
).count() == 0
assert with_derived.filter(~F.col("treatment").isin(0, 1)).count() == 0

# campaign_wave é o dense rank 0-based dos received_time distintos: verificado de
# forma independente da derivação — ordenando os pares por received_time, as
# ondas têm de sair contíguas de 0 a n-1.
wave_pairs = sorted(
    (r["received_time"], r["campaign_wave"])
    for r in with_derived.select("received_time", "campaign_wave").distinct().collect()
)
assert [wave for _, wave in wave_pairs] == list(range(len(wave_pairs)))
assert n_waves == len(wave_pairs)

# Materializa o contrato num único passe (with_derived ainda cacheado),
# persiste em disco e corta a linhagem — §8 lê de volta sem recomputar o audit.
_audit_tmp = cfg.processed_dir.parent / "_audit_contract_tmp"
contract.enforce_schema(with_derived).write.mode("overwrite").parquet(str(_audit_tmp))
with_derived.unpersist()
processed = spark.read.parquet(str(_audit_tmp)).cache()
processed.count()

print("assembly checks passed")

assembly checks passed


## 8. Contract enforcement

- `processed` foi materializado ao final da §7.1 (parquet temporário + cache).
- Imprime schema e amostra; roda os três validadores de `pipeline.validate`.


In [ ]:
n_processed = n_derived

print("=== Processed — contract ===")
print(f"rows: {n_processed:,}  |  columns: {len(contract.CONTRACT_COLUMNS)}")
print(f"grain: (account_id, offer_id, received_time)")
processed.printSchema()
processed.show(3, truncate=60)

assert processed.columns == contract.CONTRACT_COLUMNS
contract.assert_schema(processed)
contract.assert_no_unexpected_nulls(processed)
n_pydantic_sample = contract.validate_sample(processed, cfg)

print(f"contract checks passed  |  Pydantic sample: {n_pydantic_sample:,} rows")

processed.unpersist()
